# Iterate: 실행 → 관찰 → 수정

이 노트북이 출발점입니다. 에이전트가 하는 가장 보편적인 일, 즉 뭔가를 시도하고, 무슨 일이 일어났는지 읽고, 다시 시도하는 과정을 직접 해 보며 Managed Agents API의 표면을 익힙니다. 버그 두 개가 심어진 작은 패키지를 업로드하고, 테스트를 통과시키라고 지시한 뒤, 에이전트가 루프를 도는 모습을 지켜봅니다. 테스트를 돌리고, 트레이스백을 읽고, 코드를 고치고, 다시 돌리고, 초록불이 될 때까지 반복합니다.

그 과정에서 이 쿡북의 나머지가 딛고 서는 모든 API 형태를 보게 됩니다. 에이전트 / 환경 / 세션, 파일 마운트, 이벤트 스트림, 아카이브 호출입니다. 이 노트북을 마치면 에이전트를 처음부터 끝까지 구동하는 데 필요한 모든 것을 해 본 셈이 됩니다.

## 개념

알아 둘 리소스 세 가지:

- **에이전트(Agent)**: 재사용 가능한 설정(모델, 시스템 프롬프트, 도구)
- **환경(Environment)**: 컨테이너 템플릿(패키지, 네트워킹)
- **세션(Session)**: 에이전트와 환경을 묶고, 에이전트에 필요한 파일을 마운트하며, 이벤트 스트림을 만들어 냅니다

에이전트와 환경은 한 번 만들어 여러 세션에서 재사용합니다. 세션 하나는 그 자체로 완결된 실행 한 번입니다.

In [ ]:
import os
from pathlib import Path

from anthropic import Anthropic

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")

client = Anthropic()
FIXTURE = Path("example_data") / "iterate"

## 1. 에이전트 만들기

시스템 프롬프트를 일부러 성기게 뒀습니다. 단계별 대본을 따르는 대신 에이전트가 스스로 반복 루프를 알아내기를 바라기 때문입니다. 테스트 출력만으로도 할 일이 충분히 분명해서 더 손잡아 줄 필요가 없습니다.

`agent_toolset_20260401`은 내장 툴셋입니다. bash, read, write, edit, glob, grep, web_fetch, web_search가 들어 있습니다. `permission_policy`를 `always_allow`로 두면 에이전트가 확인 절차를 왕복하지 않고 바로 실행할 수 있습니다.

In [ ]:
agent = client.beta.agents.create(
    name="cookbook-iterate",
    model=MODEL,
    system=(
        "You are a debugging agent. Your job is to make failing tests pass. "
        "Run the tests, read the failures, fix the code, repeat until green. "
        "Stop when every assertion passes."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        }
    ],
)

## 2. 환경 만들기

환경은 컨테이너 템플릿입니다. `type: cloud`는 Anthropic의 호스팅 샌드박스에서 실행합니다. `networking: limited`는 임의의 외부 트래픽을 막습니다. 이 노트북은 네트워크 접근이 전혀 필요 없으므로 잠가 둡니다.

In [ ]:
env = client.beta.environments.create(
    name="cookbook-iterate-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

## 3. 실패하는 테스트 업로드하기

Files API로 파일을 업로드해 ID를 받습니다. 4단계에서 세션에 마운트합니다.

`calc.py`에는 버그 두 개가 심어져 있고 `test_calc.py`에는 이를 잡아내는 단언 세 개가 있습니다. 실패 중 하나(`test_mean`)는 나머지 둘에서 파생된 것이라, 에이전트가 지나치게 고치지 않도록 조용히 가르쳐 줍니다. `mean()`은 내부적으로 `add`와 `divide`를 호출하므로, 그 둘이 고쳐지면 `mean()`을 직접 손대지 않아도 `test_mean`이 저절로 통과하기 시작합니다.

In [ ]:
calc_file = client.beta.files.upload(
    file=("calc.py", (FIXTURE / "calc.py").read_bytes(), "text/x-python")
)
test_file = client.beta.files.upload(
    file=("test_calc.py", (FIXTURE / "test_calc.py").read_bytes(), "text/x-python")
)
print(f"uploaded: {calc_file.id}, {test_file.id}")

## 4. 세션 만들기

세션은 에이전트와 환경을 묶고, 에이전트에 필요한 파일을 마운트하고, 새 컨테이너를 시작합니다. `resources=`는 에이전트가 시작되기 전에 컨테이너에 데이터를 넣는 방법입니다. orchestrate 노트북에서는 같은 필드로 개별 파일 대신 GitHub 저장소를 클론하는 방법을 보여 줍니다.

파일은 읽기 전용인 `/mnt/session/uploads/<mount_path>` 아래에 마운트됩니다. 에이전트가 파일을 편집하려면 먼저 `/mnt/user`나 `/tmp` 같은 쓰기 가능한 디렉터리로 복사해야 하고, 나중에 가져오고 싶은 것은 `/mnt/session/outputs/`에 넣어야 합니다.

In [ ]:
session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[
        {"type": "file", "file_id": calc_file.id, "mount_path": "calc.py"},
        {"type": "file", "file_id": test_file.id, "mount_path": "test_calc.py"},
    ],
    title="Get the tests green",
)
print(f"session: {session.id}")

## 5. 에이전트를 구동하며 작업 지켜보기

두 단계입니다. 작업 내용을 담은 `user.message` 이벤트를 보내고, 에이전트가 `end_turn`에 도달할 때까지 이벤트 스트림을 읽습니다.

이 스트림은 서버 전송 이벤트(SSE) 연결입니다. 에이전트가 30초쯤 반복할 텐데 매 라운드를 실시간으로 보고 싶기 때문에 폴링 대신 이것을 씁니다(마지막 사이드바 참고).

익혀 둘 패턴 두 가지:

1. **스트림을 먼저 열고, 그다음에 보내세요.** `with` 블록이 SSE 연결을 엽니다. 블록 안에서 `send`한 것은 반드시 관찰됩니다. 열기 전에 보내면 경쟁 구간에서 발생한 이벤트를 놓칠 위험이 있습니다.
2. **`stop_reason.type == "end_turn"`인 `session.status_idle`에서 빠져나오세요.** 세션은 입력을 기다릴 때마다 유휴 상태가 되는데, 턴이 끝났을 때뿐 아니라 커스텀 도구 호출이 응답을 기다릴 때도 그렇습니다. `stop_reason.type`이 이를 구분해 주며, `end_turn`이 우리의 종료 신호입니다. gate 노트북에서 같은 루프의 `requires_action` 쪽을 보여 줍니다.

In [ ]:
with client.beta.sessions.events.stream(session.id) as stream:
    client.beta.sessions.events.send(
        session_id=session.id,
        events=[
            {
                "type": "user.message",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "The tests in /mnt/session/uploads/test_calc.py are "
                            "failing. Copy both files into /mnt/user, iterate "
                            "on calc.py until every test passes, then write the "
                            "final calc.py to /mnt/session/outputs/calc.py. "
                            "pytest isn't installed here, run the assertions "
                            "directly with `python3 -c ...` instead."
                        ),
                    }
                ],
            }
        ],
    )
    print("--- iterate loop ---")
    for ev in stream:
        match ev.type:
            case "agent.message":
                for b in ev.content:
                    if b.type == "text":
                        print(b.text, end="")
            case "agent.tool_use":
                print(f"\n[{ev.name}]")
            case "session.status_idle" if ev.stop_reason and ev.stop_reason.type == "end_turn":
                break
            case "session.status_terminated":
                break

저 `match ev.type:` 블록이 표준 스트리밍 패턴입니다. 이 쿡북의 다른 노트북들은 루프를 반복해 쓰는 대신 `utilities.py`에서 `stream_until_end_turn`으로 가져다 씁니다. 아래 검증 단계에서 그것을 사용합니다.

`wait_for_idle_status`는 `utilities.py`의 두 번째 헬퍼입니다. 7단계의 안내 상자에서 설명하는 경쟁 상태를 흡수해 줍니다. 스트림이 `session.status_idle`을 내보낸 뒤에도 세션 레코드의 서버 측 `status` 필드가 잠시 `running`으로 남아 있을 수 있고, 곧바로 `archive()`를 호출하면 400이 납니다. 이 헬퍼는 필드가 안정될 때까지 `sessions.retrieve`를 폴링할 뿐입니다. 스트리밍 직후 곧바로 아카이브하는 코드에는 이것이 필요합니다.

In [ ]:
from utilities import stream_until_end_turn, wait_for_idle_status

## 6. 검증

에이전트의 말을 그대로 믿지 마세요. 모든 단언을 한 번 더 독립적으로 실행하고 최종 `calc.py`를 출력합니다. 루프 마지막 실행과 턴 종료 사이에 에이전트가 지나치게 고쳤거나 무언가를 망가뜨렸다면 여기서 잡힙니다.

In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Re-run every assertion from "
                        "/mnt/session/uploads/test_calc.py one more time "
                        "against your final calc.py with `python3 -c ...` "
                        "to confirm they all pass, then cat the final "
                        "/mnt/session/outputs/calc.py."
                    ),
                }
            ],
        }
    ],
)
stream_until_end_turn(client, session.id)

## 7. 정리

아카이브는 세션, 환경, 에이전트를 끝났다고 표시하는 방법입니다. 실행 중인 컨테이너를 정리하고, 해당 리소스가 워크스페이스 할당량에 잡히지 않게 하며, 기본 목록 화면에서 숨깁니다. 다만 레코드와 설정, 이벤트 기록은 감사 목적이나 나중에 ID로 리소스를 조회하려는 사람을 위해 남겨 둡니다. 레코드를 완전히 없애고 싶다면 대부분의 리소스에 별도의 `delete` 엔드포인트도 있습니다(리소스 수명 주기는 operate 노트북에서 자세히 다룹니다). 하지만 실행이 끝난 시점에는 거의 항상 `archive`가 원하는 동작입니다.

In [ ]:
wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
print("archived")

## 사이드바: 스트리밍 대신 폴링

5단계의 스트리밍 패턴은 에이전트가 몇 초 이상 걸릴 작업의 진행 상황을 실시간으로 보고 싶을 때 알맞습니다. 더 짧은 작업이거나, 오래 유지되는 HTTP 연결을 두고 싶지 않은 프로덕션 코드라면 `events.list` 폴링으로 같은 일을 할 수 있습니다:

```python
client.beta.sessions.events.send(session_id=..., events=[...])
while True:
    time.sleep(2)
    events = client.beta.sessions.events.list(session.id).data
    last = events[-1] if events else None
    if last and last.type == "session.status_terminated":
        break
    if (
        last
        and last.type == "session.status_idle"
        and last.stop_reason
        and last.stop_reason.type == "end_turn"
    ):
        break
# walk events to print agent.message text
```

트레이드오프는 이렇습니다.

스트리밍은 에이전트가 일하는 모습을 지켜보고 싶을 때 유리합니다. 모든 도구 호출, 모든 부분 메시지, 모든 상태 전이가 서버가 내보내는 즉시 도착하는데, 새 워크플로를 개발할 때 딱 원하는 바입니다. 대가는 오래 유지되는 SSE 연결입니다. 프로세스가 턴이 끝날 때까지 살아 있어야 하고, 중간에 멈췄다 재개할 수 없으며, 하필 잘못된 순간의 네트워크 장애로 스트림이 끊기면 어디까지 진행했는지 깔끔하게 복구할 방법이 없습니다.

폴링은 그 반대 상황에서 유리합니다. 상태가 없고, 프로세스 재시작을 견디며, 연결을 열어 두고 싶지 않은 웹훅 핸들러나 큐 워커와 깔끔하게 어울립니다. 대가는 지연과 보이지 않는 진행 상황입니다. 다시 폴링하기 전까지는 아무것도 볼 수 없으므로 피드백이 폴링 주기에 묶이고, 긴 턴은 끝날 때까지 침묵처럼 보입니다.

에이전트가 몇 분씩 돌 수 있고 핸들러가 연결을 열어 둘 수 없는 프로덕션 환경이라면 폴링 패턴(또는 그 프로덕션 사촌 격인, gate 노트북의 `session.status_idled` 웹훅)이 필요한 것입니다.

## 다음으로 갈 곳

반복 루프는 에이전트 루프가 취할 수 있는 가장 단순한 형태입니다. 이 디렉터리의 동반 노트북 네 개가 같은 API 형태 위에서 다른 워크플로를 보여 줍니다.

- [`CMA_orchestrate_issue_to_pr.ipynb`](CMA_orchestrate_issue_to_pr.ipynb) — 더 긴 도구 체인을 거쳐 상태를 이어 가는 멀티턴 에이전트. 이슈를 읽고, 수정을 작성하고, PR을 열고, CI 실패에서 회복하고, 리뷰 코멘트에 대응하고, 병합합니다.
- [`CMA_explore_unfamiliar_codebase.ipynb`](CMA_explore_unfamiliar_codebase.ipynb) — 처음 보는 저장소에 떨어진 에이전트를 위한 근거 확보 패턴이며, 낡은 문서 함정이 심어져 있습니다. 실행 중인 세션에 컨텍스트를 더 넣는 `sessions.resources.add`도 함께 보여 줍니다.
- [`CMA_gate_human_in_the_loop.ipynb`](CMA_gate_human_in_the_loop.ipynb) — 사람 개입 워크플로를 위한 커스텀 도구 `decide()`와 `escalate()` 왕복. `requires_action` 유휴 왕복과 병렬 도구 호출 중복 제거를 다룹니다.
- [`CMA_operate_in_production.ipynb`](CMA_operate_in_production.ipynb) — 프로덕션 구성 이야기. 볼트 기반 MCP 자격 증명, 오래 유지되는 연결 없이 사람 개입을 처리하는 `session.status_idled` 웹훅, 리소스 수명 주기 CRUD 동사를 다룹니다.